# 1. KHÁM PHÁ CẤU TRÚC DANH MỤC

In [ ]:
import sys
!{sys.executable} -m pip install tqdm pandas requests numpy
import requests, json

HEADERS = {"User-Agent": "Mozilla/5.0", "Referer": "https://tiki.vn/"}

res = requests.get(
    "https://tiki.vn/api/v2/categories",
    headers=HEADERS,
    params={"include": "children", "parent_id": 1520}
)

data = res.json()
print("Status:", res.status_code)
print("Keys:", list(data.keys()))

categories = data.get("data", [])
print(f"\nSố danh mục con: {len(categories)}\n")

for cat in categories:
    print(f"[{cat['id']}] {cat['name']}  →  urlKey: {cat.get('url_key','')}")
    # In sub-category cấp 2 nếu có
    for sub in cat.get("children", []):
        print(f"    [{sub['id']}] {sub['name']}  →  urlKey: {sub.get('url_key','')}")

# 2. THIẾT LẬP THAM SỐ VÀ DANH MỤC MỤC TIÊU

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer":         "https://tiki.vn/",
    "Accept":          "application/json, text/plain, */*",
    "Accept-Language": "vi,en-US;q=0.9,en;q=0.8",
}

# Chỉ lấy category mỹ phẩm thực sự
CATEGORIES = [
    # Chăm sóc da mặt
    {"id": 1583,  "urlKey": "sua-rua-mat",                  "label": "Sữa rửa mặt"},
    {"id": 11605, "urlKey": "dung-dich-tay-trang",          "label": "Tẩy trang"},
    {"id": 2347,  "urlKey": "nuoc-hoa-hong-toner",          "label": "Toner"},
    {"id": 53350, "urlKey": "serum",                        "label": "Serum"},
    {"id": 17170, "urlKey": "kem-va-sua-duong-da",          "label": "Kem dưỡng da"},
    {"id": 1601,  "urlKey": "mat-na-cac-loai",              "label": "Mặt nạ"},
    {"id": 3422,  "urlKey": "kem-chong-nang",               "label": "Kem chống nắng (mặt)"},
    {"id": 3426,  "urlKey": "san-pham-tri-mun-va-seo",      "label": "Trị mụn & sẹo"},
    {"id": 5893,  "urlKey": "chong-lao-hoa-da",             "label": "Chống lão hóa"},
    {"id": 8206,  "urlKey": "tay-te-bao-chet",              "label": "Tẩy tế bào chết mặt"},
    {"id": 5872,  "urlKey": "xit-khoang",                   "label": "Xịt khoáng"},

    # Trang điểm
    {"id": 1587,  "urlKey": "son-trang-diem-moi",           "label": "Son môi"},
    {"id": 1585,  "urlKey": "trang-diem-mat",               "label": "Trang điểm mặt"},
    {"id": 1586,  "urlKey": "trang-diem-mat",               "label": "Trang điểm mắt"},
    {"id": 1589,  "urlKey": "dung-cu-trang-diem",           "label": "Dụng cụ trang điểm"},
    {"id": 8228,  "urlKey": "bo-trang-diem",                "label": "Bộ trang điểm"},
    {"id": 53354, "urlKey": "trang-diem-khac",              "label": "Trang điểm khác"},

    # Chăm sóc cơ thể
    {"id": 8212,  "urlKey": "tam-va-dung-cu-tam",           "label": "Sữa tắm"},
    {"id": 1610,  "urlKey": "duong-the",                    "label": "Dưỡng thể"},
    {"id": 1615,  "urlKey": "san-pham-chong-nang",          "label": "Kem chống nắng"},
    {"id": 8220,  "urlKey": "tay-te-bao-chet-co-the",       "label": "Tẩy tế bào chết body"},
    {"id": 1752,  "urlKey": "dung-dich-ve-sinh",            "label": "Dung dịch vệ sinh"},
    {"id": 3449,  "urlKey": "san-pham-tay-long",            "label": "Sản phẩm tẩy lông"},
    {"id": 17162, "urlKey": "lan-xit-khu-mui",              "label": "Lăn, xịt khử mùi"},

    # Nước hoa
    {"id": 1636,  "urlKey": "nuoc-hoa-nu",                  "label": "Nước hoa nữ"},
    {"id": 1637,  "urlKey": "nuoc-hoa-nam",                 "label": "Nước hoa nam"},

    # Chăm sóc tóc
    {"id": 8222,  "urlKey": "dau-goi-dau-xa",               "label": "Dầu gội & xả"},
    {"id": 7065,  "urlKey": "duong-toc-u-toc",              "label": "Dưỡng tóc"},
    {"id": 1620,  "urlKey": "tao-kieu-toc",                 "label": "Tạo kiểu tóc"},
    {"id": 1623,  "urlKey": "thuoc-nhuom",                  "label": "Thuốc nhuộm tóc"},
    {"id": 7061,  "urlKey": "bo-cham-soc-toc",              "label": "Bộ chăm sóc tóc"},
]

# 3. XÂY DỰNG CÁC HÀM THU THẬP VÀ TRÍCH XUẤT THÔNG TIN

In [ ]:
# Fetch 1 trang
def fetch_page(cat_id, url_key, page, limit=40):
    try:
        res = requests.get(
            "https://tiki.vn/api/v2/products",
            headers=HEADERS,
            params={
                "category_id":  cat_id,
                "urlKey":       url_key,
                "page":         page,
                "limit":        limit,
                "sort":         "top_seller",
                "version":      "home-persionalized",
                "aggregations": 2,
            },
            timeout=10,
        )
        if res.status_code != 200:
            return []
        return res.json().get("data", [])
    except Exception as e:
        print(f"  Lỗi: {e}")
        return []

# Parse 1 sản phẩm
def parse_product(p, label):
    amp         = p.get("visible_impression_info", {}).get("amplitude", {})
    badge_codes = [b.get("code","") for b in p.get("badges_new", [])]
    price       = p.get("price") or 0
    orig        = p.get("original_price") or price

    return {
        "product_id":         p.get("id"),
        "name":               p.get("name"),
        "brand_id":           p.get("brand_id"),
        "brand_name":         p.get("brand_name"),
        "seller_id":          p.get("seller_id"),
        "seller_name":        p.get("seller_name"),
        "category":           label,
        "primary_category":   p.get("primary_category_name"),
        "price":              price,
        "original_price":     orig,
        "discount_rate":      p.get("discount_rate", 0),
        "rating":             p.get("rating_average", 0),
        "review_count":       p.get("review_count", 0),
        "sold_count":         p.get("quantity_sold", {}).get("value", 0)
                              if isinstance(p.get("quantity_sold"), dict)
                              else (p.get("quantity_sold") or 0),
        # Quan trọng nhất cho bài toán
        "origin":             p.get("origin"),
        "is_imported":        amp.get("is_imported"),
        "is_authentic":       p.get("is_authentic", 0),
        "is_official_store":  p.get("is_from_official_store", False),
        "tiki_verified":      p.get("tiki_verified", 0),
        "has_authentic_badge":"authentic_brand" in badge_codes,
        "availability":       p.get("availability", 0),
    }

# 4. TIẾN HÀNH THU THẬP DỮ LIỆU

In [ ]:
# Crawl toàn bộ
all_products = []
MAX_PAGES    = 200

for cat in CATEGORIES:
    print(f"\n>>> {cat['label']} (id={cat['id']})")
    cat_count = 0

    for page in range(1, MAX_PAGES + 1):
        items = fetch_page(cat["id"], cat["urlKey"], page)

        if not items:
            print(f"Trang {page}: hết → dừng")
            break

        for p in items:
            all_products.append(parse_product(p, cat["label"]))

        cat_count += len(items)
        print(f"Trang {page}: +{len(items)} | Danh mục: {cat_count} | Tổng: {len(all_products)}")
        time.sleep(1.0)

# 5. TỔNG KẾT VÀ LƯU TRỮ DỮ LIỆU THÔ

In [ ]:
# Lưu & báo cáo nhanh
df = pd.DataFrame(all_products).drop_duplicates(subset="product_id")
path = "../data"
df.to_csv(f"{path}/tiki_cosmetics_raw.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"Tổng sản phẩm (sau dedup): {len(df):,}")
print(f"Số cột: {len(df.columns)}")
print(f"\n--- Phân bố theo danh mục ---")
print(df["category"].value_counts().to_string())
print(f"\n--- Top 15 xuất xứ ---")
print(df["origin"].value_counts().head(15).to_string())
print(f"\n--- Tỉ lệ thiếu origin ---")
print(f"{df['origin'].isna().mean():.1%}")